<div align="center">

# Week 03: The Human II: Attention, Memory and Decisions

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-03) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/Week03_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines the 40 departments of the clinic app with an illustrative share of the bookings of each. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import time
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The 40 departments of the clinic app in alphabetical order, with the share of
# bookings of each (illustrative values): six frequent departments take 70 percent.
ALL_DEPARTMENTS = [
    "Allergy and immunology", "Anaesthesiology", "Audiology", "Brain surgery",
    "Cardiology", "Cardiovascular surgery", "Chest diseases", "Child health",
    "Child psychiatry", "Dentistry", "Dermatology", "Dietetics",
    "Ear, nose and throat", "Emergency medicine", "Endocrinology", "Eye diseases",
    "Family medicine", "Gastroenterology", "General surgery", "Geriatrics",
    "Gynaecology and obstetrics", "Haematology", "Infectious diseases",
    "Internal medicine", "Medical oncology", "Nephrology", "Neurology",
    "Nuclear medicine", "Orthopaedics", "Pain medicine", "Physical therapy",
    "Plastic surgery", "Psychiatry", "Radiation oncology", "Radiology",
    "Rheumatology", "Sleep medicine", "Sports medicine", "Thoracic surgery", "Urology"]
FREQUENT = {"Family medicine": 0.20, "Internal medicine": 0.15, "Eye diseases": 0.10,
            "Ear, nose and throat": 0.09, "Orthopaedics": 0.08, "Dermatology": 0.08}
OTHERS = [d for d in ALL_DEPARTMENTS if d not in FREQUENT]
DEPARTMENTS = {d: FREQUENT.get(d, 0.30 / len(OTHERS)) for d in ALL_DEPARTMENTS}
print("Setup complete. Running in Colab:", IN_COLAB, "| departments:", len(DEPARTMENTS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 3.1": (lambda: float((times_s > 0.6).mean()), 1e-6),
    "Exercise 3.2": (lambda: float(np.polyfit([2, 4, 6, 8], [0.61, 0.83, 1.02, 1.25], 1)[0]), 1e-6),
    "Exercise 3.3": (lambda: float(np.polyval(np.polyfit([2, 4, 6, 8], [0.61, 0.83, 1.02, 1.25], 1), 12)), 1e-6),
    "Exercise A": (lambda: 0.40 + 0.25 * (40 + 1) / 2, 1e-6),
    "Exercise B": (lambda: 0.4 * np.log2(1 / 0.4) + 0.3 * np.log2(1 / 0.3) + 0.2 * np.log2(1 / 0.2) + 0.1 * np.log2(1 / 0.1), 1e-6),
    "Exercise C": (lambda: 1 / (1 + np.exp((11 - 7) / 0.8)), 1e-6),
    "Exercise D": (lambda: menu_time("grouped", 0.2), 1e-6),
    "Exercise E": (lambda: float(STAGE3["split, novice"]), 1e-6),
}

## Python step 3: Random numbers, timing and a first fit

People never take exactly the same time twice, so studies of attention and memory measure many trials and summarise them. This step draws random numbers that behave like reaction times and measures the duration of a piece of code. It then fits a straight line through points with NumPy &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-2)&#93;. The line answers the question of the overview: How much longer does a search take for each additional item &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-1)&#93;?

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "DEPARTMENTS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Random numbers

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(7)        # a generator with a fixed seed
print("five numbers between 0 and 1:", rng.random(5).round(3))
lamps = rng.integers(1, 9, size=10)     # which of eight lamps lights in ten trials
print("lamps:", lamps)
print("a random department:", rng.choice(["Cardiology", "Neurology", "Dermatology"]))
rng2 = np.random.default_rng(7)
print("same seed, same numbers:", rng2.random(5).round(3))

A random generator produces numbers that look random but follow from a starting value, the *seed*. The same seed always gives the same numbers, so a simulation can be repeated exactly. The method `random` draws numbers between 0 and 1, and `integers(1, 9)` draws whole numbers from 1 to 8 (the upper limit is excluded). The method `choice` picks an element of a list. Experiments use such numbers to decide which lamp lights or which item is the target, so that a participant cannot guess the order.

### Simulated reaction times

In [ ]:
# Python practice: a small example for learning the language
times_s = rng.lognormal(mean=np.log(0.45), sigma=0.25, size=200)
print(f"mean {times_s.mean():.3f} s | median {np.median(times_s):.3f} s | "
      f"standard deviation {times_s.std():.3f} s")
print("fastest and slowest:", times_s.min().round(3), times_s.max().round(3))
print("10th and 90th percentiles:", np.percentile(times_s, [10, 90]).round(3))
plt.hist(times_s, bins=25, color="#5A4BB5", edgecolor="white")
plt.xlabel("reaction time (s)")
plt.ylabel("trials")
plt.title("200 simulated reaction times")
plt.show()

Reaction times are positive and have a long tail of slow trials, so they are often simulated with a *lognormal* distribution, whose logarithm is normally distributed. The mean is pulled up by the slow trials, and the median, the middle value, is not. The standard deviation measures the spread, and `np.percentile` returns the values below which a given share of the trials lies. The histogram counts the trials in 25 bins.

### Measuring the duration of code

In [ ]:
# Python practice: a small example for learning the language
start = time.perf_counter()
ordered = np.sort(rng.random(1_000_000))       # sort one million numbers
elapsed = time.perf_counter() - start
print(f"sorting took {1000 * elapsed:.1f} ms")
print("the smallest three:", ordered[:3].round(6))

`time.perf_counter()` reads a precise clock in seconds. The difference of two readings is the duration of the code between them. A reaction time is measured the same way: The clock starts when the stimulus appears and stops at the response. The interactive lab of this week measures your responses with the same kind of clock in the browser.

### A straight line through points

In [ ]:
# Python practice: a small example for learning the language
n_items = np.array([4, 8, 16, 32])            # items on a screen
mean_s = np.array([0.62, 0.83, 1.24, 2.07])     # mean search time for each
slope, intercept = np.polyfit(n_items, mean_s, 1)
print(f"time = {intercept:.3f} + {slope:.4f} x items  (seconds)")
print(f"each additional item costs {1000 * slope:.1f} ms")
x = np.linspace(0, 36, 50)
plt.scatter(n_items, mean_s, color="#C98510", zorder=3, label="measured means")
plt.plot(x, intercept + slope * x, color="#5A4BB5", label="fitted line")
plt.xlabel("items on the screen")
plt.ylabel("mean search time (s)")
plt.legend()
plt.show()

`np.polyfit(x, y, 1)` finds the straight line, a polynomial of degree 1, that passes closest to the points in the sense of least squares. It returns the slope first and the intercept second. The slope is the extra time per additional item and the intercept the time that does not depend on the number of items. The function `np.linspace(0, 36, 50)` gives 50 evenly spaced values for drawing the line. Week 4 looks at what least squares means.

**Quick check.** Two generators are created with `np.random.default_rng(7)`. Why do they give the same numbers?

<details><summary>Answer</summary>

The numbers follow from the seed. The same seed starts the same sequence, so the two generators agree, which makes simulations repeatable.

</details>

### Exercises for Python step 3

**Exercise 3.1.** Store the share of the simulated reaction times `times_s` that are longer than 0.6 s in `share_slow`.

In [ ]:
share_slow = None   # your answer

check("Exercise 3.1", share_slow)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
share_slow = (times_s > 0.6).mean()

check("Exercise 3.1", share_slow)

**Exercise 3.2.** Mean times of 0.61, 0.83, 1.02 and 1.25 s were measured with 2, 4, 6 and 8 items. Fit a straight line with `np.polyfit` and store its slope, in seconds per item, in `ms_per_item`.

In [ ]:
ms_per_item = None   # your answer

check("Exercise 3.2", ms_per_item)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
ms_per_item = np.polyfit([2, 4, 6, 8], [0.61, 0.83, 1.02, 1.25], 1)[0]

check("Exercise 3.2", ms_per_item)

**Exercise 3.3.** Use the same fit to predict the time with 12 items and store it in `t_12`. The function `np.polyval(coefficients, x)` evaluates a fitted line at `x`.

In [ ]:
t_12 = None   # your answer

check("Exercise 3.3", t_12)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
coef = np.polyfit([2, 4, 6, 8], [0.61, 0.83, 1.02, 1.25], 1)
t_12 = np.polyval(coef, 12)

check("Exercise 3.3", t_12)

## 1. Visual search: Your search slopes &#91;[3](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-3)&#93;

*Lecture: [Attention and visual search](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#attention-and-visual-search)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "times_s" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A search experiment produces hundreds of reaction times in eight cells: Two kinds of search, four numbers of items, target present or absent. Python groups them, fits a line for every combination and compares the slopes, which is the quantity that tells efficient and inefficient search apart.

**From the formulas to the code.** The number of items $N$ is `N`, the base time $a$ is `a` and the time to inspect one item $c$ is `c`. The function `search_time` returns $a + c(N + 1)/2$ for a present target and $a + cN$ for an absent one. A fitted slope $b$ is the first value that `np.polyfit` returns.

In [ ]:
def search_time(N, a=0.40, c=0.05, present=True):
    """Expected time of a serial, self-terminating search through N items in s."""
    N = np.asarray(N, float)
    return a + c * (N + 1) / 2 if present else a + c * N

print("worked example, 16 icons: present", search_time(16), "s | absent", search_time(16, present=False), "s")
print("40 department names, present:", search_time(40, c=0.25), "s")

**Python note.** A function with the keyword argument `present=True` can be called with `present=False` to switch the formula. The expression `x if condition else y` chooses one of two values in a single line.

**Exercise A.** A patient searches the alphabetical list of 40 department names for one name. With $a = 0.40$ s and $c = 0.25$ s per name, store the expected time for a present target in `t_names`, with the formula and without calling `search_time`.

In [ ]:
t_names = None   # your answer

check("Exercise A", t_names)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_names = 0.40 + 0.25 * (40 + 1) / 2

check("Exercise A", t_names)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_SEARCH`. Left empty, the variable makes the code use a simulated class of 24 students. Their times are assumptions of this Colab notebook: Feature search is nearly flat, and conjunction search inspects about 45 ms per item.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_SEARCH = """
"""

def simulated_search_class(n_students=24, seed=RANDOM_STATE):
    """Two trials per cell, with assumed slopes for feature and conjunction search."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        speed = rng.lognormal(0, 0.15)
        for cond, c in [("feature", 0.004), ("conjunction", 0.045)]:
            for N in (4, 8, 16, 32):
                for present in (1, 1, 0, 0):
                    mean = search_time(N, 0.45, c, bool(present)) * speed
                    rt = mean * rng.lognormal(0, 0.15)
                    ok = int(rng.random() > 0.04)
                    rows.append([student, cond, N, present, ok, round(1000 * rt)])
    return pd.DataFrame(rows, columns=["student", "condition", "N", "present",
                                       "correct", "rt_ms"])

if MY_SEARCH.strip():
    search = pd.read_csv(io.StringIO(MY_SEARCH.strip()))
    search["student"] = 1
    source = "your own trials from the interactive lab"
else:
    search = simulated_search_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(search), "trials")
ok = search[search["correct"] == 1]
cells = ok.groupby(["condition", "present", "N"])["rt_ms"].mean().unstack("N")
print((cells / 1000).round(3))

**Python note.** Only correct trials enter the means, because a wrong answer may come from a guess. `unstack("N")` turns the numbers of items into columns, so that each row of the table is one combination of condition and presence.

In [ ]:
slopes = {}
for (cond, present), d in ok.groupby(["condition", "present"]):
    means = d.groupby("N")["rt_ms"].mean() / 1000
    b, a = np.polyfit(means.index, means.values, 1)
    slopes[(cond, "present" if present else "absent")] = (a, b)
    print(f"{cond:<11} {'present' if present else 'absent ':<7} "
          f"time = {a:.3f} + {1000 * b:5.1f} ms x N")
for cond in ("feature", "conjunction"):
    ratio = slopes[(cond, "absent")][1] / slopes[(cond, "present")][1]
    print(f"{cond}: absent slope / present slope = {ratio:.2f}")

**Python note.** Grouping by two columns gives one group per combination, and the loop unpacks the pair of keys `(cond, present)` directly. A dictionary with tuples as keys keeps the intercept and the slope of every line.

In [ ]:
# Figure: search time against the number of items, with the fitted lines
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
x = np.array([0, 34])
for k, cond in enumerate(["feature", "conjunction"]):
    for present, style in [(1, "-"), (0, "--")]:
        d = ok[(ok["condition"] == cond) & (ok["present"] == present)]
        m = d.groupby("N")["rt_ms"].mean() / 1000
        a, b = slopes[(cond, "present" if present else "absent")]
        label = "present" if present else "absent"
        ax[k].plot(m.index, m.values, "o", color="#5A4BB5" if present else "#C98510")
        ax[k].plot(x, a + b * x, style, color="#5A4BB5" if present else "#C98510",
                   label=f"{label}: {1000 * b:.0f} ms per item")
    ax[k].set_title(f"{cond.capitalize()} search")
    ax[k].set_xlabel("number of items N")
    ax[k].legend(loc="upper left")
ax[0].set_ylabel("mean time of correct trials (s)")
plt.tight_layout()
plt.show()

**Python note.** `sharey=True` gives both panels the same vertical axis, so the slopes can be compared by eye. The line style `"--"` draws a dashed line for the absent targets.

**Observe and reflect.** In the conjunction search, is the absent slope close to twice the present slope, as a serial search that stops at the target predicts? What would a ratio close to 1 mean?

## 2. Hick-Hyman: Your rate of gain of information &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-5)&#93;

*Lecture: [Choosing among options: The Hick-Hyman law](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#the-hick-hyman-law)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "slopes" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Forty choice reactions in four blocks become four mean times. A line through them on a logarithmic axis gives your rate of gain of information in bits per second. The same few lines compute the information of any distribution of choices, such as the bookings of forty departments.

**From the formulas to the code.** In the code, $n$ is `n`, $a$ is `a` and $b$ is `b`, and $\log_2(n + 1)$ is `np.log2(n + 1)`. The function `information` returns $H = \sum_i p_i \log_2(1/p_i)$ for a list of probabilities `p`.

In [ ]:
def hick_time(n, a=0.20, b=0.15):
    """Hick: choice time for n equally likely options, in s."""
    return a + b * np.log2(np.asarray(n, float) + 1)

def information(p):
    """Hyman: information H in bits of a choice with probabilities p."""
    p = np.asarray(p, float)
    p = p[p > 0]
    return float((p * np.log2(1 / p)).sum())

print(f"n = 8: {hick_time(8):.3f} s | n = 40: {hick_time(40):.3f} s")
print("H of 0.5, 0.25, 0.125, 0.125:", information([0.5, 0.25, 0.125, 0.125]), "bits")
H_dep = information(list(DEPARTMENTS.values()))
print(f"H of the 40 departments: {H_dep:.2f} bits (equal shares: {np.log2(40):.2f} bits)")

**Python note.** `p[p > 0]` drops the options with probability zero, whose term in the sum is zero anyway but would make `np.log2` divide by zero. `list(DEPARTMENTS.values())` takes the shares out of the dictionary.

**Exercise B.** Four departments receive 40, 30, 20 and 10 percent of the bookings. Store the information of the choice in bits in `H_four`, computed with the formula of Hyman rather than with `information`.

In [ ]:
H_four = None   # your answer

check("Exercise B", H_four)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
H_four = (0.4 * np.log2(1 / 0.4) + 0.3 * np.log2(1 / 0.3)
          + 0.2 * np.log2(1 / 0.2) + 0.1 * np.log2(1 / 0.1))

check("Exercise B", H_four)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_LAMPS`. Left empty, the variable makes the code use a simulated class of 24 students with assumed values of $a$ and $b$.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_LAMPS = """
"""

def simulated_lamps_class(n_students=24, seed=RANDOM_STATE):
    """Ten trials per block, with personal values of a and b."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        a, b = rng.normal(0.22, 0.03), rng.normal(0.13, 0.025)
        for n in (1, 2, 4, 8):
            for trial in range(1, 11):
                rt = hick_time(n, a, b) * rng.lognormal(0, 0.12)
                ok = int(rng.random() > 0.01 * n)
                rows.append([student, n, trial, 0, ok, round(1000 * rt)])
    return pd.DataFrame(rows, columns=["student", "n", "trial", "practice", "correct",
                                       "rt_ms"])

if MY_LAMPS.strip():
    lamps_df = pd.read_csv(io.StringIO(MY_LAMPS.strip()))
    lamps_df["student"] = 1
    source = "your own trials from the interactive lab"
else:
    lamps_df = simulated_lamps_class()
    source = "a simulated class of 24 students"
real = lamps_df[(lamps_df["practice"] == 0) & (lamps_df["correct"] == 1)]
fits = []
for student, d in real.groupby("student"):
    m = d.groupby("n")["rt_ms"].mean() / 1000
    b, a = np.polyfit(np.log2(m.index + 1), m.values, 1)
    fits.append([student, a, b, 1 / b])
fits = pd.DataFrame(fits, columns=["student", "a_s", "b_s_per_bit", "bits_per_s"])
print("Data:", source)
print(fits.drop(columns="student").describe().loc[["mean", "min", "max"]].round(3))
errors = lamps_df[lamps_df["practice"] == 0].groupby("n")["correct"].mean()
print("share of correct presses by n:", errors.round(3).to_dict())

**Python note.** The fit uses `np.log2(m.index + 1)` as the horizontal values, so the straight line of `np.polyfit` is the Hick-Hyman law. Its slope is $b$, and $1/b$ is the rate of gain of information.

In [ ]:
# Figure: mean choice time against the information of the choice
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
means = real.groupby(["student", "n"])["rt_ms"].mean().unstack("n") / 1000
bits = np.log2(np.array(means.columns) + 1)
for _, row in means.iterrows():
    ax[0].plot(bits, row.values, color="#C9C3E6", lw=1)
ax[0].plot(bits, means.mean().values, "o-", color="#5A4BB5", lw=2.5, label="class mean")
ax[0].set_xticks(bits, [f"n={n}\n{v:.2f} bits" for n, v in zip(means.columns, bits)])
ax[0].set_ylabel("mean choice time (s)")
ax[0].set_title("Choice time against log2(n + 1)")
ax[0].legend()
ax[1].hist(fits["bits_per_s"], bins=10, color="#C9C3E6", edgecolor="white")
ax[1].set_xlabel("rate of gain of information 1/b (bits per second)")
ax[1].set_ylabel("students")
ax[1].set_title("Rates of the class")
plt.tight_layout()
plt.show()

**Python note.** `set_xticks(positions, labels)` places the ticks at the bits and writes two lines under each, the number of lamps and the information, joined by `\n`.

**Observe and reflect.** Did your errors grow with the number of lamps? A participant who answers faster but less accurately trades accuracy for speed. How would such a trade change the fitted slope $b$?

## 3. Memory: Your span with and without chunks &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-8)&#93;

*Lecture: [Memory: Holding and recalling](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#memory)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "fits" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Fourteen codes of seven lengths and two formats give a curve of recall against length for each format. A fit of that curve turns your answers into a single number, the span, and shows how much grouping the digits helped.

**From the formulas to the code.** In the code, the length $L$ is `L`, the span $L_{50}$ is `L50` and the spread $s$ is `s`. The function `p_recall` returns $1/(1 + e^{(L - L_{50})/s})$, and `fit_span` finds $L_{50}$ and $s$ by maximum likelihood on a grid, as Week 2 found the reading threshold.

In [ ]:
def p_recall(L, L50=7.0, s=0.8):
    """Probability of recalling a code of length L correctly."""
    return 1 / (1 + np.exp((np.asarray(L, float) - L50) / s))

def fit_span(L, correct):
    """Maximum-likelihood L50 and s of p_recall, found on a grid."""
    L50s, ss = np.linspace(2, 14, 241), np.linspace(0.2, 4, 39)
    A, S = np.meshgrid(L50s, ss, indexing="ij")
    p = 1 / (1 + np.exp((np.asarray(L, float) - A[..., None]) / S[..., None]))
    p = np.clip(p, 1e-6, 1 - 1e-6)
    y = np.asarray(correct, float)
    ll = (y * np.log(p) + (1 - y) * np.log(1 - p)).sum(axis=-1)
    i, j = np.unravel_index(np.argmax(ll), ll.shape)
    return L50s[i], ss[j]

print(f"11 digits, plain: {p_recall(11):.4f} | grouped (L50 = 9, s = 1): "
      f"{p_recall(11, 9, 1.0):.3f}")

**Python note.** The grid search is the same as in Week 2, with a decreasing curve: The longer the code, the lower the probability. `A[..., None]` adds an axis so that one line evaluates all codes for all candidate pairs.

**Exercise C.** With $L_{50} = 7$ and $s = 0.8$, store the probability of recalling an 11-digit patient number correctly in `p_eleven`, computed with the formula.

In [ ]:
p_eleven = None   # your answer

check("Exercise C", p_eleven)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_eleven = 1 / (1 + np.exp((11 - 7) / 0.8))

check("Exercise C", p_eleven)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_CODES`. The code always adds a simulated class of 24 students, whose spans are assumptions of this Colab notebook. Your own codes appear as student 0.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_CODES = """
"""

def simulated_codes_class(n_students=24, seed=RANDOM_STATE):
    """One code per length and format, with personal spans."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        span = rng.normal(7.0, 0.8)
        for fmt, bonus in [("plain", 0.0), ("chunked", 1.4)]:
            for L in range(4, 11):
                ok = int(rng.random() < p_recall(L, span + bonus, 0.7))
                rows.append([student, L, fmt, ok])
    return pd.DataFrame(rows, columns=["student", "length", "format", "correct"])

codes = simulated_codes_class()
if MY_CODES.strip():
    mine = pd.read_csv(io.StringIO(MY_CODES.strip())).assign(student=0)
    codes = pd.concat([mine[codes.columns], codes])
spans = []
for (student, fmt), d in codes.groupby(["student", "format"]):
    L50, s = fit_span(d["length"], d["correct"])
    spans.append([student, fmt, L50, s])
spans = pd.DataFrame(spans, columns=["student", "format", "L50", "s"])
print(spans.groupby("format")["L50"].describe()[["mean", "min", "max"]].round(2))
if 0 in set(spans["student"]):
    print(spans[spans["student"] == 0].round(2).to_string(index=False))
pooled = {fmt: fit_span(d["length"], d["correct"]) for fmt, d in codes.groupby("format")}
for fmt, (L50, s) in pooled.items():
    print(f"curve of the whole class, {fmt}: L50 = {L50:.2f}, s = {s:.2f}")

**Python note.** With only one code per length and format, a single student's fit is rough. The fit of the pooled answers of the whole class, `pooled`, describes the class as a population, and its spread $s$ also contains the differences between students. `assign(student=0)` marks your own rows before `pd.concat` stacks them on the class.

In [ ]:
# Figure: share of correct codes by length, with the curve of the class means
L = np.arange(3, 12.01, 0.05)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for fmt, col in [("plain", "#C98510"), ("chunked", "#5A4BB5")]:
    share = codes[codes["format"] == fmt].groupby("length")["correct"].mean()
    L50, s = pooled[fmt]
    ax.plot(share.index, share.values, "o", color=col)
    ax.plot(L, p_recall(L, L50, s), color=col, label=f"{fmt}: span L50 = {L50:.1f}")
ax.axvline(11, color="#B53F2E", ls=":", lw=1.5)
ax.text(11.1, 0.85, "patient\nnumber", color="#B53F2E", fontsize=9)
ax.set_xlabel("length of the code L (digits)")
ax.set_ylabel("share recalled correctly")
ax.legend()
plt.tight_layout()
plt.show()

**Python note.** The dotted vertical line marks the 11 digits of the patient number, so the figure shows directly how far beyond the span the kiosk asks patients to go.

**Observe and reflect.** Did grouping the digits raise your own span? Which other codes do apps ask people to copy from one place to another, and how could each be avoided?

## 4. Menus: From search to decision &#91;[12](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#ref-12)&#93;

*Lecture: [Menus: From search to decision](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#menus-from-search-to-decision)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "spans" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The menu model joins the formulas of Sections 1 and 2 for three designs and any level of experience. Written as one function, it answers what-if questions in a loop and draws the crossover between designs that a table of single numbers would hide.

**From the formulas to the code.** The level of experience $e$ is `e`, and the search time $T_s$ is `search_time` with $c = 0.25$ s per name. The decision time $T_h$ is `hick_time`, and the pointing time $T_p$ is `T_POINT`. For the split menu, `P_TOP` is the share of choices found among the six frequent departments at the top.

In [ ]:
T_POINT, P_TOP, C_NAME = 1.0, 0.70, 0.25

def menu_time(design, e):
    """Expected selection time in s of a department, for experience e from 0 to 1."""
    S = lambda n: search_time(n, 0.40, C_NAME)
    if design == "flat":                 # one alphabetical list of 40
        return (1 - e) * S(40) + e * hick_time(40) + T_POINT
    if design == "grouped":              # 5 groups, then 8 departments
        return (1 - e) * (S(5) + S(8)) + e * (hick_time(5) + hick_time(8)) + 2 * T_POINT
    if design == "split":                # 6 frequent on top, then all 40
        novice = P_TOP * S(6) + (1 - P_TOP) * (0.40 + C_NAME * (6 + 41 / 2))
        expert = P_TOP * hick_time(6) + (1 - P_TOP) * hick_time(40)
        return (1 - e) * novice + e * expert + T_POINT
    raise ValueError(design)

for design in ("flat", "grouped", "split"):
    print(f"{design:<8} novice {menu_time(design, 0):.3f} s | patient (e = 0.2) "
          f"{menu_time(design, 0.2):.3f} s | expert {menu_time(design, 1):.3f} s")

**Python note.** `S = lambda n: ...` defines a short function inside `menu_time`, so the search formula with its fixed parameters is written once. A function that raises `ValueError` for an unknown design reports a typing mistake at once instead of returning a wrong number.

**Exercise D.** Store the expected time of the grouped menu for a typical patient with $e = 0.2$ in `t_grouped`.

In [ ]:
t_grouped = None   # your answer

check("Exercise D", t_grouped)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_grouped = menu_time("grouped", 0.2)

check("Exercise D", t_grouped)

In [ ]:
# Figure: selection time against experience, and the crossover
e = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for design, col in [("flat", "#8A93A3"), ("grouped", "#C98510"), ("split", "#5A4BB5")]:
    t = np.array([menu_time(design, v) for v in e])
    ax.plot(e, t, color=col, lw=2.2, label=design)
flat = np.array([menu_time("flat", v) for v in e])
grouped = np.array([menu_time("grouped", v) for v in e])
cross = e[np.argmin(np.abs(flat - grouped))]
ax.axvline(cross, color="#B53F2E", ls=":", lw=1.5)
ax.text(cross + 0.02, 5.7, f"flat = grouped\nat e = {cross:.2f}", fontsize=9, color="#B53F2E")
ax.set_xlabel("experience e (0 = novice, 1 = expert)")
ax.set_ylabel("expected selection time (s)")
ax.legend()
plt.tight_layout()
plt.show()

**Python note.** A list comprehension evaluates `menu_time` for each of the 101 levels of experience. `np.argmin(np.abs(flat - grouped))` finds the level where the two curves are closest, which is where they cross.

**Explore.** Move the sliders and watch the three designs change. The bars show the three times. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W

def explore(e=0.2, c=0.25, p_top=0.70, t_point=1.0):
    global C_NAME, P_TOP, T_POINT
    old = (C_NAME, P_TOP, T_POINT)
    C_NAME, P_TOP, T_POINT = c, p_top, t_point
    designs = ("flat", "grouped", "split")
    times = [menu_time(d, e) for d in designs]
    C_NAME, P_TOP, T_POINT = old
    for d, t in zip(designs, times):
        print(f"{d:<8} {t:6.2f} s")
    plt.figure(figsize=(5, 2.6))
    plt.barh(designs, times, color=["#7A8790", "#5A4BB5", "#0E7A78"])
    plt.xlabel(f"expected selection time at e = {e:.2f} (s)")
    plt.xlim(0, max(10, 1.1 * max(times)))
    plt.tight_layout()
    plt.show()

W.interact(explore,
           e=W.FloatSlider(value=0.2, min=0, max=1, step=0.05, description="e"),
           c=W.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01, description="c (s)"),
           p_top=W.FloatSlider(value=0.70, min=0, max=1, step=0.05, description="share top"),
           t_point=W.FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description="T_p (s)"));

**Python note.** `global` lets `explore` change the three parameters that `menu_time` reads. The line after the computation puts the old values back, so the rest of the Colab notebook is not affected by the sliders.

**Observe and reflect.** The grouped menu helps novices and slows experts. Which users of the clinic app are novices and which are experts, and which design would you give each?

## 5. The clinic app, stage 3

*Lecture: [The clinic app, stage 3: Menus and forms that need less memory](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#the-clinic-app-stage-3)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "menu_time" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The stage compares designs on three questions at once: Time, memory and defaults. A dictionary of results keeps every number of the stage together, and Week 9 tests the same designs with simulated patients.

In [ ]:
STAGE3 = {}
for design in ("flat", "grouped", "split"):
    STAGE3[f"{design}, novice"] = round(menu_time(design, 0.2), 3)
    STAGE3[f"{design}, nurse"] = round(menu_time(design, 1.0), 3)
STAGE3["recall 11 digits, plain"] = round(float(p_recall(11, *pooled["plain"])), 4)
STAGE3["recall 11 digits, grouped 3-4-4"] = round(float(p_recall(11, *pooled["chunked"])), 4)
N_DAY, KEEP, WRONG = 600, 0.80, 0.25      # bookings, keep the default, default is wrong
STAGE3["unwanted defaults per day"] = N_DAY * KEEP * WRONG
for k, v in STAGE3.items():
    print(f"{k:<34} {v}")

**Python note.** `p_recall(11, *pooled["plain"])` unpacks the two fitted values of the class, $L_{50}$ and $s$, into the arguments of the function. The star works with any sequence of values in the right order.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
designs = ["flat", "grouped", "split"]
xpos = np.arange(3)
ax[0].bar(xpos - 0.2, [STAGE3[f"{d}, novice"] for d in designs], 0.4, color="#C98510",
          label="patient, e = 0.2")
ax[0].bar(xpos + 0.2, [STAGE3[f"{d}, nurse"] for d in designs], 0.4, color="#5A4BB5",
          label="nurse, e = 1")
ax[0].set_xticks(xpos, designs)
ax[0].set_ylabel("expected time to choose a department (s)")
ax[0].set_title("Choosing a department")
ax[0].legend()
labels = ["typed from memory", "typed, grouped 3-4-4", "card scanned"]
values = [STAGE3["recall 11 digits, plain"], STAGE3["recall 11 digits, grouped 3-4-4"], 1.0]
ax[1].bar(labels, values, color=["#B53F2E", "#C98510", "#2E8B57"])
ax[1].set_ylabel("probability of a correct patient number")
ax[1].set_title("Entering the patient number")
plt.tight_layout()
plt.show()

**Python note.** Two sets of bars side by side are drawn by shifting the positions by 0.2 to the left and to the right and making each bar 0.4 wide.

**Exercise E.** Store the expected time of the split menu for a typical patient, as kept in `STAGE3`, in `t_split`.

In [ ]:
t_split = None   # your answer

check("Exercise E", t_split)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_split = STAGE3["split, novice"]

check("Exercise E", t_split)

**Observe and reflect.** The split menu wins for patients and for the nurse. What happens to it when the frequent departments change, for example in the flu season, and how could the app keep the top section right?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE3" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-03/"
               "NB03_attention_memory_decisions.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below has a menu with its own size and its own frequent items. The function of Section 4 compares the three designs for any of them after one change of a name.

The menus below are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {"clinic": (40, 0.70), "banking": (14, 0.80), "e_government": (60, 0.55),
        "food_delivery": (120, 0.40), "course_registration": (200, 0.30)}

def app_times(n, p_top, e):
    """Flat, grouped and split menus of n items for experience e, in s."""
    S = lambda k: search_time(k, 0.40, C_NAME)
    g = int(round(np.sqrt(n)))                       # about sqrt(n) groups
    k = int(np.ceil(n / g))                          # items per group
    flat = (1 - e) * S(n) + e * hick_time(n) + T_POINT
    grouped = (1 - e) * (S(g) + S(k)) + e * (hick_time(g) + hick_time(k)) + 2 * T_POINT
    novice = p_top * S(6) + (1 - p_top) * (0.40 + C_NAME * (6 + (n + 1) / 2))
    expert = p_top * hick_time(6) + (1 - p_top) * hick_time(n)
    split = (1 - e) * novice + e * expert + T_POINT
    return {"flat": flat, "grouped": grouped, "split": split}

n, p_top = APPS[APP]
for e in (0.0, 0.2, 1.0):
    t = app_times(n, p_top, e)
    best = min(t, key=t.get)
    print(f"{APP}, {n} items, e = {e}: " + ", ".join(f"{k} {v:.2f} s" for k, v in t.items())
          + f" -> best: {best}")

**Python note.** `min(t, key=t.get)` returns the key of the dictionary whose value is smallest, here the fastest design. About $\sqrt{n}$ groups of about $\sqrt{n}$ items keep both levels short.

**Observe and reflect.** For which app does grouping beat the split menu for novices? What does that say about menus with many items and few frequent ones?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | An integrated development environment offers 300 commands in menus. Use `menu_time` with your own numbers to compare a flat command palette with grouped menus. |
| Web development | A navigation bar has 9 items, 3 of them used in 80 percent of visits. Compute the information of the choice and the expected expert time with the formula of Hyman. |
| Mobile development | A one-time code of 6 digits is sent by text message. Compute its probability of correct recall with the fitted span, and explain why phones now fill such codes automatically. |
| Game development | A game asks players to choose one of 12 weapons in a radial menu. Simulate 20 players with the Hick-Hyman law and compare a menu of 12 with two rings of 6. |
| Embedded systems and the Internet of Things | A washing machine has 15 programmes on one knob. Model the choice time of a novice with visual search and propose a grouping. |
| Data science | Fit the visual search slopes of Section 1 for each student separately and report the mean and the spread of the conjunction slopes. |
| Cybersecurity | Passwords that must be remembered exceed the span of working memory. Compute $p$ for random passwords of 8 and 12 characters with $L_{50} = 6$, and discuss passphrases as chunks. |
| Health informatics | A nurse is interrupted 30 times in a shift, and each interruption costs 20 s to resume. Compute the lost time per shift and propose a design that shortens resumption. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-03).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>